# KG Construction v2 — v_n fact layer
Builds the Neo4j graph from the **v_n (gpt-oss-20b, 2-call)** fact layer. No judges; 53-term concept vocabulary from `concept_vocab.json`.

**Put these in Drive root (`MyDrive/`) first:** `kg_v2_utils.py`, `clean_fact_layer_v2.py`, `populate_kg_v2.py`, `concept_vocab.json`, `compute_similar_facts.py` (unchanged old file), and your merged v_n folder `MERGED_v_n/` (flat folder of all chunk case JSONs; do not include `monitoring/` or `checkpoints/`).

Run order: 1 → 5 (5 is a dry run), then 6 → 12. Stages are separate cells so a Colab disconnect only costs one stage (all writes are MERGE = safe to re-run).
Use a **CPU runtime** for cells 1–10; GPU only for cell 11.

In [1]:
!pip install -q neo4j tqdm sentence-transformers faiss-cpu

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
D = '/content/drive/MyDrive'
for f in ['kg_v2_utils_v2.py','clean_fact_layer_v2.py','populate_kg_v2_v2.py','concept_vocab.json','compute_similar_facts.py']:
    ok = os.path.exists(f'{D}/KG/{f}'); print('OK     ' if ok else 'MISSING', f)
    if ok: shutil.copy(f'{D}/KG/{f}', f'/content/{f}')
V_N_DIR = f'{D}/MERGED_fact_layer_final_v2'
CLEAN_DIR = f'{D}/CLEANED_v2'
n = len([x for x in os.listdir(V_N_DIR) if x.endswith('.json')]) if os.path.isdir(V_N_DIR) else 0
print('v_n JSON files:', n)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK      kg_v2_utils_v2.py
OK      clean_fact_layer_v2.py
OK      populate_kg_v2_v2.py
OK      concept_vocab.json
OK      compute_similar_facts.py
v_n JSON files: 9987


## 3 — Credentials (never hardcode; the old notebook leaked a password)
Aura's bolt username is always `neo4j` (the old notebook had the instance id there by mistake).

In [3]:
import getpass, os
os.environ['NEO4J_URI'] = input('Aura URI (neo4j+s://xxxx.databases.neo4j.io): ').strip()
os.environ['NEO4J_USER'] = 'e600e96a'
os.environ['NEO4J_PASSWORD'] = getpass.getpass('Aura password: ')
from neo4j import GraphDatabase
drv = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=('e600e96a', os.environ['NEO4J_PASSWORD']))
drv.verify_connectivity()
with drv.session() as s:
    print('Connected. Existing nodes:', s.run('MATCH (n) RETURN count(n) AS n').single()['n'],
          '| relationships:', s.run('MATCH ()-[r]->() RETURN count(r) AS n').single()['n'])
drv.close()

Aura URI (neo4j+s://xxxx.databases.neo4j.io): neo4j+s://e600e96a.databases.neo4j.io
Aura password: ··········
Connected. Existing nodes: 0 | relationships: 0


## 4 — Clean the v_n JSONs (no LLM, no Neo4j). Writes `CLEANED_v2/` and `_report.json`

In [14]:
!python /content/clean_fact_layer_v2.py "{V_N_DIR}" "{CLEAN_DIR}"

{
 "statute_kind": {
  "section": 30783,
  "constitution": 9482,
  "act_ref": 9710
 },
 "cite_type": {
  "other": 11789,
  "reporter": 49361,
  "named": 17350,
  "case_number": 9281
 },
 "out_of_vocab": {},
 "flags": {
  "outcome_conflict": 241,
  "petitioner_args_missing": 210,
  "outcome_unknown": 218
 },
 "bare_dropped": 36839,
 "empty_concepts": 1356,
 "cases": 9986
}


## 5 — DRY RUN (builds every row set, writes nothing). Check the counts look sane first.

In [24]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --dry-run 2>&1 | grep -v 'batch '

  Loaded 9986 cleaned cases

── nodes ──
  Case: 9986
  Statute: 19234
  Topic: 14
  Party: 13768
  LegalConcept (all vocab terms): 53
  Argument: 45332

── rels ──
  INVOLVES: 51088
  RELATES_TO: 9986
  HAS_PARTY: 19938
  MAKES_ARGUMENT: 45332
  APPLIES_CONCEPT: 22086

── citations ──
  CITES: 2597
   {'skipped_other': 11789, 'skipped_reporter': 49361, 'name_too_generic': 3586, 'rejected_future_target': 48784, 'unresolved_named': 11167, 'skipped_case_number': 9281, 'resolved': 2597}
  NOTE: chunk-only graph resolves only citations whose target is in the loaded set; hand-check ~50 edges (match_score is stored).


## 6 — Constraints & indexes

In [25]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage constraints

  Loaded 9986 cleaned cases


## 7 — Smoke test on 200 cases (recommended on a fresh instance)

In [26]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage nodes --sample 200
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage rels  --sample 200

  Loaded 200 cleaned cases

── nodes ──
    Case: batch 1/1
  Case: 200
    Statute: batch 1/2
  Statute: 709
    Topic: batch 1/1
  Topic: 9
    Party: batch 1/1
  Party: 338
    LegalConcept (all vocab terms): batch 1/1
  LegalConcept (all vocab terms): 53
    Argument: batch 1/2
  Argument: 700
  Loaded 200 cleaned cases

── rels ──
    INVOLVES: batch 1/2
  INVOLVES: 898
    RELATES_TO: batch 1/1
  RELATES_TO: 200
    HAS_PARTY: batch 1/1
  HAS_PARTY: 400
    MAKES_ARGUMENT: batch 1/2
  MAKES_ARGUMENT: 700
    APPLIES_CONCEPT: batch 1/1
  APPLIES_CONCEPT: 425


## 8 — Nodes (full)

In [27]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage nodes

  Loaded 9986 cleaned cases

── nodes ──
    Case: batch 1/20
  Case: 9986
    Statute: batch 1/39
    Statute: batch 21/39
  Statute: 19234
    Topic: batch 1/1
  Topic: 14
    Party: batch 1/28
    Party: batch 21/28
  Party: 13768
    LegalConcept (all vocab terms): batch 1/1
  LegalConcept (all vocab terms): 53
    Argument: batch 1/91
    Argument: batch 21/91
    Argument: batch 41/91
    Argument: batch 61/91
    Argument: batch 81/91
  Argument: 45332


## 9 — Relationships (full)

In [28]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage rels

  Loaded 9986 cleaned cases

── rels ──
    INVOLVES: batch 1/103
    INVOLVES: batch 21/103
    INVOLVES: batch 41/103
    INVOLVES: batch 61/103
    INVOLVES: batch 81/103
    INVOLVES: batch 101/103
  INVOLVES: 51088
    RELATES_TO: batch 1/20
  RELATES_TO: 9986
    HAS_PARTY: batch 1/40
    HAS_PARTY: batch 21/40
  HAS_PARTY: 19938
    MAKES_ARGUMENT: batch 1/91
    MAKES_ARGUMENT: batch 21/91
    MAKES_ARGUMENT: batch 41/91
    MAKES_ARGUMENT: batch 61/91
    MAKES_ARGUMENT: batch 81/91
  MAKES_ARGUMENT: 45332
    APPLIES_CONCEPT: batch 1/45
    APPLIES_CONCEPT: batch 21/45
    APPLIES_CONCEPT: batch 41/45
  APPLIES_CONCEPT: 22086


## 10 — Citations (name-based; each edge stores `match_score` — hand-check ~50)

In [29]:
!python -u /content/populate_kg_v2_v2.py --folder "{CLEAN_DIR}" --stage citations

  Loaded 9986 cleaned cases

── citations ──
    CITES: batch 1/6
  CITES: 2597
   {'skipped_other': 11789, 'skipped_reporter': 49361, 'name_too_generic': 3586, 'rejected_future_target': 47921, 'unresolved_named': 11167, 'skipped_case_number': 9281, 'resolved': 2597}
  NOTE: chunk-only graph resolves only citations whose target is in the loaded set; hand-check ~50 edges (match_score is stored).


## 11 — SIMILAR_FACTS
Uses the unchanged `compute_similar_facts.py` (reads `issue` + `facts` from the cleaned files; URI/password come from env vars). New cache file names so v0 embeddings are not reused. For GPU: switch runtime, `!pip install -q faiss-gpu`, drop `--cpu`.

In [30]:
EMB = f'{D}/embeddings_vn.npy'; PAIRS = f'{D}/similar_pairs_vn.json'
!python -u /content/compute_similar_facts.py --folder "{CLEAN_DIR}" --threshold 0.75 --emb-path "{EMB}" --pairs-path "{PAIRS}" --cpu

Loaded 9966 cases with embedding text
Loading cached embeddings from /content/drive/MyDrive/embeddings_vn.npy
Loading cached similar pairs from /content/drive/MyDrive/similar_pairs_vn.json

Connecting to Neo4j at neo4j+s://e600e96a.databases.neo4j.io ...
Connected.

Creating SIMILAR_FACTS edges (12324 pairs, both directions) ...
  [  10000/12324]  6.0s
  [  12324/12324]  7.4s
  Done. 12324 SIMILAR_FACTS pairs (24648 directed edges)

Done. SIMILAR_FACTS graph layer is ready.
This enables the symbolic reasoning engine's precedent traversal
via SIMILAR_FACTS and FOLLOWS relationships.


## 12 — Verify

In [31]:
from neo4j import GraphDatabase
import os
drv = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=('e600e96a', os.environ['NEO4J_PASSWORD']))
with drv.session() as s:
    print('NODES  ', s.run('MATCH (n) RETURN labels(n)[0] AS l, count(n) AS n ORDER BY n DESC').data())
    print('RELS   ', s.run('MATCH ()-[r]->() RETURN type(r) AS t, count(r) AS n ORDER BY n DESC').data())
    print('Outcome', s.run('MATCH (c:Case) RETURN c.outcome AS o, count(*) AS n ORDER BY n DESC').data())
    print('Flagged cases', s.run('MATCH (c:Case) WHERE size(c.quality_flags)>0 RETURN count(c) AS n').single()['n'])
    print('Cases with a resolved statute', s.run("MATCH (c:Case)-[:INVOLVES]->(s:Statute) WHERE s.statute_type IN ['section','constitution'] RETURN count(DISTINCT c) AS n").single()['n'])
    print('Art 19/20/22 statute ids', [r['id'] for r in s.run("MATCH (s:Statute) WHERE s.statute_id =~ 'Constitution_Art_(19|20|22)([(].*)?' RETURN s.statute_id AS id ORDER BY id")])
    for r in s.run('MATCH (a:Case)-[r:CITES]->(b:Case) RETURN a.title AS src, r.citation_raw AS cite, b.title AS tgt, r.match_score AS score ORDER BY score LIMIT 10'): print('CITES', dict(r))
drv.close()

NODES   [{'l': 'Argument', 'n': 45332}, {'l': 'Statute', 'n': 19234}, {'l': 'Party', 'n': 13768}, {'l': 'Case', 'n': 9986}, {'l': 'LegalConcept', 'n': 53}, {'l': 'Topic', 'n': 14}]
RELS    [{'t': 'INVOLVES', 'n': 51088}, {'t': 'MAKES_ARGUMENT', 'n': 45332}, {'t': 'SIMILAR_FACTS', 'n': 24648}, {'t': 'APPLIES_CONCEPT', 'n': 22086}, {'t': 'HAS_PARTY', 'n': 19938}, {'t': 'RELATES_TO', 'n': 9986}, {'t': 'CITES', 'n': 2586}]
Outcome [{'o': 'Accept', 'n': 5319}, {'o': 'Reject', 'n': 3322}, {'o': 'Partially Accept', 'n': 589}, {'o': 'Directions Issued', 'n': 538}, {'o': 'Unknown', 'n': 218}]
Flagged cases 660
Cases with a resolved statute 8524
Art 19/20/22 statute ids ['Constitution_Art_19', 'Constitution_Art_19(1)', 'Constitution_Art_19(1)(2)', 'Constitution_Art_19(1)(a)', 'Constitution_Art_19(1)(b)', 'Constitution_Art_19(1)(c)', 'Constitution_Art_19(1)(c)(g)', 'Constitution_Art_19(1)(d)', 'Constitution_Art_19(1)(e)', 'Constitution_Art_19(1)(f)', 'Constitution_Art_19(1)(g)', 'Constitution_Art

## DANGER ZONE — wipe graph (deliberate rebuild only)
Disabled by default; set `CONFIRM` to the exact phrase to enable. Never run on an instance holding a graph you want to keep.

In [34]:
CONFIRM = ''   # to wipe, set to: 'WIPE THIS GRAPH'
if CONFIRM == 'WIPE THIS GRAPH':
    from neo4j import GraphDatabase
    import os
    d = GraphDatabase.driver(os.environ['NEO4J_URI'], auth=('e600e96a', os.environ['NEO4J_PASSWORD']))
    with d.session() as s:
        while s.run('MATCH (n) WITH n LIMIT 10000 DETACH DELETE n RETURN count(*) AS c').single()['c']: pass
    d.close(); print('Graph wiped.')
else:
    print('Wipe disabled.')

Wipe disabled.


In [35]:
!python /content/rule_smoke_test.py \
    /content/Rules_article_19_20_22_v5_fixed.json \
    /content/rules_art21_32_14_v2_fixed.json


Outcome mix by year (whole graph):
  2000: n= 392 Accept 53.3%  Reject 38.0%
  2001: n= 399 Accept 46.6%  Reject 44.1%
  2002: n= 397 Accept 51.4%  Reject 35.5%
  2003: n= 396 Accept 49.0%  Reject 42.9%
  2004: n= 394 Accept 49.5%  Reject 38.1%
  2005: n= 392 Accept 51.0%  Reject 36.7%
  2006: n= 395 Accept 56.5%  Reject 32.9%
  2007: n= 399 Accept 50.4%  Reject 42.1%
  2008: n= 400 Accept 58.0%  Reject 32.0%
  2009: n=  26 Accept 38.5%  Reject 30.8%
  2010: n= 400 Accept 46.5%  Reject 38.8%
  2011: n= 400 Accept 54.2%  Reject 34.5%
  2012: n= 399 Accept 48.9%  Reject 34.1%
  2013: n= 399 Accept 50.4%  Reject 32.3%
  2014: n= 400 Accept 54.2%  Reject 28.5%
  2015: n= 398 Accept 55.5%  Reject 32.2%
  2016: n= 400 Accept 53.0%  Reject 32.2%
  2017: n= 400 Accept 52.5%  Reject 31.0%
  2018: n= 400 Accept 57.2%  Reject 26.8%
  2019: n= 400 Accept 53.8%  Reject 30.8%
  2020: n= 400 Accept 47.8%  Reject 35.2%
  2021: n= 400 Accept 55.5%  Reject 28.8%
  2022: n= 400 Accept 61.0%  Reject 24.2